# 0. Imports & Reproducibility

In [183]:
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import community as community_louvain
import faiss
import networkx as nx
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from gensim.models import Word2Vec
from node2vec import Node2Vec
from torch.utils.data import DataLoader, Dataset

In [184]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 1. Download and Store Dataset

In [185]:
def ingest_reddit_data(
    subreddit_key: str, n_rows: int = 1_000_000, force_rerun: bool = False
) -> Path:
    """
    Orchestrates the ETL process for a specific subreddit's comment data.

    Args:
        subreddit_key: Dictionary key from 'splits' (e.g., 'changemyview').
        n_rows: Maximum records to process for the local sample.
        force_rerun: If True, bypasses existence check and overwrites existing parquet file.

    Returns:
        Path to the processed Parquet file.
    """
    out_path = Path(f"data/processed/{subreddit_key}_sample.parquet")

    # Idempotency check: Skip heavy network I/O if the target file is already present
    if out_path.exists() and not force_rerun:
        print(f"Skipping ingestion: Local cache found at {out_path}")
        return out_path

    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Define schema subset based on downstream analytical requirements
    feature_cols = [
        "author",
        "body",
        "created_utc",
        "id",
        "link_id",
        "name",
        "parent_id",
        "score",
        "controversiality",
        "total_awards_received",
    ]
    splits = {
        "changemyview": "data/changemyview-*-of-*.parquet",
    }

    print(f"Streaming data from HuggingFace for: r/{subreddit_key}...")

    # Execute lazy-evaluated ETL pipeline
    try:
        (
            pl.scan_parquet(
                f"hf://datasets/HuggingFaceGECLM/REDDIT_comments/{splits[subreddit_key]}"
            )
            .select(feature_cols)
            # Filter out deleted/removed content to maintain high data quality for NLP tasks
            .filter(~pl.col("body").is_in(["[deleted]", "[removed]"]))
            .limit(n_rows)
            # Stream directly to disk using ZSTD to balance compression ratio and write speed
            .sink_parquet(out_path, compression="zstd")
        )
        print(f"Successfully wrote {n_rows} rows to {out_path}")
    except KeyError:
        raise ValueError(f"Subreddit '{subreddit_key}' not found in defined splits.")
    except Exception as e:
        print(f"Pipeline failed: {e}")
        raise

    return out_path


# --- Execution Control ---
# Toggle 'force_rerun' if the upstream data schema changes or a larger sample is needed
OUT = ingest_reddit_data("changemyview", n_rows=1_000_000, force_rerun=False)

Skipping ingestion: Local cache found at data/processed/changemyview_sample.parquet


# 2. Load Data from Parquet File

In [186]:
df = (
    # Scan the metadata and define the lazy query plan
    pl.scan_parquet("data/processed/changemyview_sample.parquet")
    # Constrain sample size for rapid local prototyping
    .head(100000)
    # Trigger execution and load into memory
    .collect()
    # Bridge to Pandas for ecosystem compatibility
    .to_pandas()
)

# 3. Create Train and Test Dataset

### 3.1 Data Preprocessing, Temporal Splitting & Metadata Mapping

In [187]:
# --- 1. Data Cleaning & Type Casting ---

# Ensure text integrity by removing null observations in the primary feature
df = df.dropna(subset=["body"])

# Filter out anonymous/deleted accounts to maintain attribution quality
df = df[df["author"] != "[deleted]"]

# Normalize timestamps: Convert raw strings to numeric Unix seconds, then to datetime objects
# 'coerce' handles malformed strings by returning NaT, preventing pipeline crashes
df["created_utc"] = pd.to_numeric(df["created_utc"], errors="coerce")
df["date"] = pd.to_datetime(df["created_utc"], unit="s", errors="coerce")


# --- 2. Temporal Train/Test Split ---

# Use a temporal 80/20 split rather than a random shuffle to prevent 'look-ahead' bias.
# This simulates a real-world scenario where we predict future comments based on past data.
cutoff = df["created_utc"].quantile(0.8)

df_train = df[df["created_utc"] <= cutoff].copy()
df_test = df[df["created_utc"] > cutoff].copy()


# --- 3. Metadata Mapping (Lookup Tables) ---

# Create lightweight author lookups for efficient O(1) retrieval.
# Mappings are scoped strictly within splits to enforce isolation and prevent leakage.
id2author_train = df_train.set_index("id")["author"].to_dict()
id2author_test = df_test.set_index("id")["author"].to_dict()

### 3.2 Interaction Network Construction

In [188]:
def build_reply_pairs(df_split, id2author):
    """
    Constructs a positive interaction dataset by mapping comments to their parent authors.
    Filters for comment-to-comment replies and removes self-interactions.
    """
    # Reddit 'parent_id' prefixes: t1 = Comment, t3 = Link/Post.
    # We restrict analysis to comment-to-comment interactions to capture conversational dynamics.
    parent_comment_ids = df_split["parent_id"].astype(str)
    is_comment_reply = parent_comment_ids.str.startswith("t1_")
    df_r = df_split[is_comment_reply].copy()

    # Extract the raw 36-base ID by stripping the 't1_' type prefix for join compatibility
    df_r["parent_key"] = df_r["parent_id"].str.replace("^t1_", "", regex=True)

    # Resolve parent author identities via the provided lookup table (O(1) mapping)
    df_r["parent_author"] = df_r["parent_key"].map(id2author)

    # --- Data Integrity & Quality Filtering ---
    # 1. Drop replies where the parent comment falls outside the current split (boundary integrity)
    df_r = df_r.dropna(subset=["parent_author"])
    # 2. Exclude self-replies to ensure we only model interpersonal interactions
    df_r = df_r[df_r["author"] != df_r["parent_author"]]

    # Feature selection and renaming to standard (u, v) graph notation
    pairs_pos = df_r[
        ["author", "parent_author", "created_utc", "link_id", "id", "parent_key"]
    ].copy()
    pairs_pos = pairs_pos.rename(
        columns={
            "author": "u",
            "parent_author": "v",
            "id": "u_comment_id",
            "parent_key": "v_comment_id",
        }
    )

    # Label as positive instances for downstream binary classification
    pairs_pos["y"] = 1
    return pairs_pos


# Generate interaction sets; scoped within splits to prevent data leakage
pos_train = build_reply_pairs(df_train, id2author_train)
pos_test = build_reply_pairs(df_test, id2author_test)

print(f"Positive samples - Train: {len(pos_train):,} | Test: {len(pos_test):,}")

Positive samples - Train: 45,452 | Test: 10,214


In [189]:
# --- Graph Diagnostics: Sparsity & Degree Distribution ---

# Calculate the ratio of users who engaged in at least one reply
pos_users = set(pos_train["u"]) | set(pos_train["v"])
all_users = set(df_train["author"].dropna().unique())
print(
    f"Engagement Coverage: {len(pos_users)} / {len(all_users)} users with interactions"
)

# Analyze the 'Out-Degree' (number of replies sent per user)
print("\nReplies per user statistics:")
print(pos_train.groupby("u").size().describe())

Engagement Coverage: 6622 / 8152 users with interactions

Replies per user statistics:
count    6068.000000
mean        7.490442
std        25.103788
min         1.000000
25%         1.000000
50%         2.000000
75%         6.000000
max      1009.000000
dtype: float64


### 3.3 Negative Sampling Strategy

In [190]:
def build_hard_negatives(df_split, pos_pairs, k_per_pos=2, seed=42):
    """
    Generates 'hard' negative samples for link prediction by identifying potential
    interactions that did NOT occur within the same discussion thread context.
    """
    # Initialize a BitGenerator for reproducible stochastic sampling
    rng = np.random.default_rng(seed)

    # 1) Contextual Mapping: Identify all active participants per discussion thread (link_id).
    # This defines our 'closed-world' candidate pool for each observation.
    thread_users = (
        df_split.groupby("link_id")["author"].apply(lambda s: set(s.dropna())).to_dict()
    )

    # 2) Network Topology: Extract existing interaction edges in (u, v) space.
    # We treat edges as symmetric to prevent sampling reciprocal replies as negatives, which would introduce label noise.
    reply_edges = set(zip(pos_pairs["u"], pos_pairs["v"]))
    reply_edges_sym = reply_edges | {(v, u) for (u, v) in reply_edges}

    neg_rows = []
    # Project to minimal feature set to reduce overhead during iteration
    pos_pairs_small = pos_pairs[["u", "v", "link_id"]].copy()

    for u, v, link_id in pos_pairs_small.itertuples(index=False):
        users = list(thread_users.get(link_id, []))
        if len(users) <= 1:
            continue

        # Candidate Filtering:
        # Target users in the same thread (high-signal 'hard' negatives) excluding the source 'u'
        cand = [x for x in users if x != u]
        if not cand:
            continue

        # Collision Avoidance: Remove candidates where a ground-truth interaction (u, x) exists
        cand = [x for x in cand if (u, x) not in reply_edges_sym]
        if not cand:
            continue

        # Stochastic Sampling: Select 'k' negatives per positive to maintain class ratio
        take = min(k_per_pos, len(cand))
        sampled = rng.choice(cand, size=take, replace=False)

        for x in sampled:
            neg_rows.append((u, x, link_id, 0))

    return pd.DataFrame(neg_rows, columns=["u", "v", "link_id", "y"])

### 3.4 Triplet Dataset Assembly

In [191]:
def build_triplets_from_hard_negatives(pos_pairs, neg_pairs, seed=42):
    """
    Constructs triplets (u, v_pos, v_neg) for metric learning.
    For each positive interaction (u, v_pos), sample one hard negative v_neg
    from the same thread context.
    """
    rng = np.random.default_rng(seed)

    # Map u → list of negative candidates
    neg_map = neg_pairs.groupby("u")["v"].apply(list).to_dict()

    triplets = []

    for u, v_pos, link_id in pos_pairs[["u", "v", "link_id"]].itertuples(index=False):
        neg_candidates = neg_map.get(u, [])
        if not neg_candidates:
            continue

        # Sample one hard negative for this positive
        v_neg = rng.choice(neg_candidates)

        triplets.append((u, v_pos, v_neg, link_id))

    return pd.DataFrame(triplets, columns=["u", "v_pos", "v_neg", "link_id"])


# --- Generate hard negatives (unchanged) ---
neg_train = build_hard_negatives(df_train, pos_train, k_per_pos=1)
neg_test = build_hard_negatives(df_test, pos_test, k_per_pos=1)

# --- Build triplets ---
triplets_train = build_triplets_from_hard_negatives(pos_train, neg_train)
triplets_test = build_triplets_from_hard_negatives(pos_test, neg_test)

print("Triplets Train:", len(triplets_train))
print("Triplets Test:", len(triplets_test))
print(triplets_train.head())

Triplets Train: 45259
Triplets Test: 10001
                 u                 v_pos            v_neg    link_id
0        Jaberkaty  Thompson_S_Sweetback  ancillarynipple  t3_16ralh
1  ancillarynipple  Thompson_S_Sweetback        Jaberkaty  t3_16ralh
2  ancillarynipple  Thompson_S_Sweetback        Jaberkaty  t3_16ralh
3           llatia             gchase723          banebot  t3_16s6jg
4     cardswsbound                 nix0n         Rongoose  t3_16rzx1


In [192]:
def test_triplet_integrity(triplets, pos_df, neg_df):
    """
    Comprehensive suite to verify triplet logic and data leakage.
    """
    # 1. Structural Check
    assert not triplets.isnull().values.any(), "Triplets contain NaN values"

    # 2. Contextual Integrity: v_neg must actually exist in the negative pool for that user
    # This ensures rng.choice didn't pull a random user from the wrong context
    u_to_negs = neg_df.groupby("u")["v"].apply(set).to_dict()

    for row in triplets.itertuples():
        # Check: v_pos and v_neg must be different
        assert row.v_pos != row.v_neg, f"Anchor {row.u} has identical Pos/Neg target"

        # Check: anchor cannot be its own target
        assert row.u != row.v_pos, f"Self-loop found in positive: {row.u}"
        assert row.u != row.v_neg, f"Self-loop found in negative: {row.u}"

        # Check: v_neg must be a valid 'hard' negative from the pool
        valid_negs = u_to_negs.get(row.u, set())
        assert row.v_neg in valid_negs, (
            f"User {row.v_neg} is not a valid hard negative for {row.u}"
        )

    # 3. Label Leakage: Ensure v_neg is NEVER a real positive for that user
    # (Symmetric check to be extra safe)
    pos_edges = set(zip(pos_df["u"], pos_df["v"]))
    pos_edges_sym = pos_edges | {(v, u) for (u, v) in pos_edges}

    triplet_neg_edges = set(zip(triplets["u"], triplets["v_neg"]))
    overlap = triplet_neg_edges.intersection(pos_edges_sym)

    assert len(overlap) == 0, (
        f"Leakage detected! {len(overlap)} 'negatives' are actually real interactions."
    )

    print("✅ All Triplet Integrity Tests Passed!")


# Run the test
test_triplet_integrity(triplets_train, pos_train, neg_train)

✅ All Triplet Integrity Tests Passed!


### 3.5 User Textual Profile Construction

In [193]:
# --- 1. Corpus Preparation & Leakage Prevention ---

# Isolate training and testing text to ensure that future comments do not
# influence the historical representations of users in the training set.
df_train_text = df_train.dropna(subset=["body", "id"]).copy()
df_test_text = df_test.dropna(subset=["body", "id"]).copy()

# --- 2. Temporal Aggregation (Feature Engineering) ---

# Construct a profile for each author.
# We join the most recent comments to capture the user's current interests/voice.
user_text_train = (
    df_train_text.sort_values(
        "created_utc"
    )  # Enforce chronology to correctly identify the 'tail'
    .groupby("author")["body"]
    # Hyperparameter: Concatenating the last 10 comments balances context vs. sequence length
    .apply(lambda s: " ".join(s.tail(10)))
)

user_text_test = (
    df_test_text.sort_values("created_utc")
    .groupby("author")["body"]
    .apply(lambda s: " ".join(s.tail(10)))
)

# Convert to hash maps (dict) for O(1) lookup performance during the mapping phase
user_text_dict_train = user_text_train.to_dict()
user_text_dict_test = user_text_test.to_dict()

# --- 3. Coverage Analysis (Data Integrity Check) ---

# Quantify the 'Cold-Start' issue: users in the interaction pairs who lack
# textual history. Significant missingness here indicates a sampling mismatch.
missing_train = triplets_train["u"].map(user_text_dict_train).isna().mean()
print(f"Missing text profile ratio (Train - Source User): {missing_train:.2%}")

missing_test = triplets_test["u"].map(user_text_dict_test).isna().mean()
print(f"Missing text profile ratio (Test - Source User): {missing_test:.2%}")

Missing text profile ratio (Train - Source User): 0.00%
Missing text profile ratio (Test - Source User): 0.00%


### 3.6 Attach Text to Triplets

In [194]:
# --- 1. Prepare Data Containers ---

# Create copies to prevent SettingWithCopy warnings and isolate split changes
triplets_train = triplets_train.copy()
triplets_test = triplets_test.copy()


def attach_text(triplets, user_text_dict):
    """Adds historical text for source (u) and target (v) users."""

    # Map text profiles to user IDs
    triplets["text_u"] = triplets["u"].map(user_text_dict)
    triplets["text_v_pos"] = triplets["v_pos"].map(user_text_dict)
    triplets["text_v_neg"] = triplets["v_neg"].map(user_text_dict)

    # Remove observations missing text for either user to ensure a complete feature set
    return triplets.dropna(subset=["text_u", "text_v_pos", "text_v_neg"])


# --- 2. Execute Merge & Cleanup ---

triplets_train_txt = attach_text(triplets_train, user_text_dict_train)
triplets_test_txt = attach_text(triplets_test, user_text_dict_test)

# --- 3. Progress Check ---

# Log row counts to monitor data loss during the mapping/dropping process
print(f"Train Retention: {len(triplets_train):,} -> {len(triplets_train_txt):,}")
print(f"Test Retention:  {len(triplets_test):,} -> {len(triplets_test_txt):,}")

Train Retention: 45,259 -> 45,259
Test Retention:  10,001 -> 10,001


### 3.7 Save Train and Test Datasets

In [195]:
triplets_train_txt.to_parquet("data/processed/train_triplets_txt.parquet", index=False)
triplets_test_txt.to_parquet("data/processed/test_triplets_txt.parquet", index=False)

# 4.  Implement Baselines

### 4.1 Random Baseline

In [196]:
# Candidates = all users from training
all_users_train = set(pos_train["u"]).union(set(pos_train["v"]))
all_users_train = list(all_users_train)


def recommend_random(user, k=10, exclude_seen=True):
    """
    Recommend k random users.

    Args:
        user: source user
        k: number of recommendations
        exclude_seen: avoid recommending already interacted users (train)
    """
    # Case 1: No filtering of historical interactions
    # Only exclude self-recommendations
    if not exclude_seen:
        candidates = [u for u in all_users_train if u != user]
        return random.sample(candidates, k)

    # Case 2: Exclude users already interacted with in training
    # (prevents recommending known neighbors)
    seen = set(pos_train[pos_train["u"] == user]["v"].values)

    # Remove self and previously interacted users
    candidates = [
        u for u in all_users_train
        if u != user and u not in seen
    ]

    # If candidate pool is smaller than k,
    # return all available candidates
    if len(candidates) < k:
        return candidates

    # Uniform random sampling without replacement
    return random.sample(candidates, k)

### 4.2 Common Neighbor Baseline

In [197]:
# Build an Adjacency List from the TRAIN set
# We treat the network as undirected to find "friends of friends" (mutual interactors)
adj = defaultdict(set)
for u, v in zip(pos_train["u"], pos_train["v"]):
    adj[u].add(v)
    adj[v].add(u)


def recommend_common_neighbors(user, k=10, exclude_seen=True):
    """
    Recommend users based on the number of shared interaction partners (Common Neighbors).

    Args:
        user: The source user for whom to generate recommendations.
        k: Number of recommendations to return.
        exclude_seen: If True, prevents recommending users already interacted with in training.
    """
    if user not in adj:
        # Cold-start: If the user has no history, no neighbors can be found
        return []

    user_neighbors = adj[user]
    candidate_scores = defaultdict(int)

    # Traverse to neighbors (friends) and then to their neighbors (friends of friends)
    for neighbor in user_neighbors:
        for fof in adj[neighbor]:  # Friend of a Friend
            if fof != user:
                # Increment score for every shared path (common neighbor)
                candidate_scores[fof] += 1

    # Filter: Remove users the target user has already interacted with in the training set
    if exclude_seen:
        for seen_user in user_neighbors:
            if seen_user in candidate_scores:
                del candidate_scores[seen_user]

    # Sort candidates by the number of common neighbors in descending order
    sorted_recs = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)
    return [rec[0] for rec in sorted_recs[:k]]

### 4.3 Popularity Baseline

In [198]:
# 1. Compute popularity scores from TRAIN only
popularity_scores = pos_train["v"].value_counts().to_dict()

# 2. Global ranking of users by popularity
global_pop_ranking = sorted(
    popularity_scores.keys(), key=lambda x: popularity_scores[x], reverse=True
)


def recommend_popularity(user, k=10, exclude_seen=True):
    """
    Recommend top-k most popular users.

    Args:
        user: source user
        k: number of recommendations
        exclude_seen: avoid recommending already interacted users (train)
    """
    
    # Case 1: No novelty constraint
    # Simply return the globally most popular users
    if not exclude_seen:
        return global_pop_ranking[:k]

    # Case 2: Enforce novelty by excluding:
    # - The user itself
    # - Users already interacted with during training
    seen = set(pos_train[pos_train["u"] == user]["v"].values)

    recs = []
    for candidate in global_pop_ranking:

        # Prevent self-recommendation
        if candidate == user:
            continue

        # Prevent recommending already known neighbors
        if candidate in seen:
            continue

        recs.append(candidate)

        # Stop once k valid recommendations are collected
        if len(recs) == k:
            break

    return recs

# 5. Siamese CNN for Text-Based User Embeddings

### 5.1 Text Encoding

In [199]:
# Define a simple regex to extract word-level tokens (alphabetic only)
TOKEN_RE = re.compile(r"[A-Za-z']+")


def tokenize(text: str):
    """Lowercases and extracts valid word tokens from raw text."""
    return TOKEN_RE.findall(text.lower())


# Constraints for memory efficiency and noise reduction
MAX_VOCAB = 50_000
MIN_FREQ = 2

# Build frequency distribution from training corpus only to prevent leakage
counter = Counter()
for t in triplets_train_txt["text_u"].tolist():
    counter.update(tokenize(t))
for t in triplets_train_txt["text_v_pos"].tolist():
    counter.update(tokenize(t))
for t in triplets_train_txt["text_v_neg"].tolist():
    counter.update(tokenize(t))

# Reserved tokens for sequence padding and out-of-vocabulary terms
PAD = "<pad>"
UNK = "<unk>"

# Initialize vocabulary with reserved indices
vocab = {PAD: 0, UNK: 1}

# Populate vocabulary with the most frequent terms meeting the frequency threshold
for w, c in counter.most_common(MAX_VOCAB):
    if c < MIN_FREQ:
        break
    vocab[w] = len(vocab)

pad_id = vocab[PAD]
unk_id = vocab[UNK]

print(f"Final Vocab Size: {len(vocab):,}")

Final Vocab Size: 46,978


In [200]:
# Fixed sequence length to ensure uniform input dimensions for the model
MAX_LEN = 256


def encode(text: str):
    """
    Converts raw text into a list of integer IDs.
    Unknown words are mapped to 'unk_id' and sequences are truncated to MAX_LEN.
    """
    ids = [vocab.get(w, unk_id) for w in tokenize(text)]
    return ids[:MAX_LEN]

### 5.2 Dataset Definition

In [201]:
class TripletDataset(Dataset):
    """
    Custom PyTorch Dataset to serve (u, v_pos, v_neg) triplets, their respective
    encoded histories, and the binary interaction label.
    """

    def __init__(self, df_triplets):
        self.u = df_triplets["u"].tolist()
        self.v_pos = df_triplets["v_pos"].tolist()
        self.v_neg = df_triplets["v_neg"].tolist()
        self.u_texts = df_triplets["text_u"].tolist()
        self.v_pos_texts = df_triplets["text_v_pos"].tolist()
        self.v_neg_texts = df_triplets["text_v_neg"].tolist()

    def __len__(self):
        return len(self.u)

    def __getitem__(self, idx):
        # Returns raw IDs and encoded text sequences for the given index
        return (
            self.u[idx],
            self.v_pos[idx],
            self.v_neg[idx],
            encode(self.u_texts[idx]),
            encode(self.v_pos_texts[idx]),
            encode(self.v_neg_texts[idx]),
        )

### 5.3 Batch Collation (Dynamic Padding)

In [202]:
def collate_fn(batch):
    """
    Dynamic padding: Aligns sequences within a batch to the length of
    the longest sequence found in that specific batch.
    """
    # Unpack columns from the batch of tuples
    u, v_pos, v_neg, u_seqs, v_pos_seqs, v_neg_seqs = zip(*batch)

    # Track original lengths for masking or sequence packing
    u_lens = torch.tensor([len(s) for s in u_seqs], dtype=torch.long)
    v_pos_lens = torch.tensor([len(s) for s in v_pos_seqs], dtype=torch.long)
    v_neg_lens = torch.tensor([len(s) for s in v_neg_seqs], dtype=torch.long)

    # Determine batch-wide maximum dimensions
    max_u = max(u_lens).item()
    max_v_pos = max(v_pos_lens).item()
    max_v_neg = max(v_neg_lens).item()

    # Initialize tensors filled with the PAD token
    u_tensor = torch.full((len(batch), max_u), pad_id, dtype=torch.long)
    v_pos_tensor = torch.full((len(batch), max_v_pos), pad_id, dtype=torch.long)
    v_neg_tensor = torch.full((len(batch), max_v_neg), pad_id, dtype=torch.long)

    # Copy sequence data into the padded containers
    for i, s in enumerate(u_seqs):
        u_tensor[i, : len(s)] = torch.tensor(s, dtype=torch.long)

    for i, s in enumerate(v_pos_seqs):
        v_pos_tensor[i, : len(s)] = torch.tensor(s, dtype=torch.long)

    for i, s in enumerate(v_neg_seqs):
        v_neg_tensor[i, : len(s)] = torch.tensor(s, dtype=torch.long)

    return (
        list(u),
        list(v_pos),
        list(v_neg),
        u_tensor,
        v_pos_tensor,
        v_neg_tensor,
        u_lens,
        v_pos_lens,
        v_neg_lens,
    )

### 5.4 Data Loader Initilization

In [203]:
# Number of samples processed before the model updates its internal parameters
BATCH_SIZE = 128

# Instantiate dataset objects for training and evaluation
train_ds = TripletDataset(triplets_train_txt)
test_ds = TripletDataset(triplets_test_txt)

# Training Loader: Shuffle enabled to prevent the model from learning the order of samples
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    generator=torch.Generator().manual_seed(SEED),
)

# Testing Loader: Shuffle disabled to ensure consistent, reproducible evaluation
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    generator=torch.Generator().manual_seed(SEED),
)

### 5.5 Create Siamese CNN

##### 5.5.1 Create Text Encoder to build user embeddings

In [204]:
class TextCNNEncoder(nn.Module):
    """
    CNN-based text encoder that maps a sequence of token IDs
    to a fixed-size L2-normalized embedding.

    Uses multiple kernel sizes to capture n-gram features
    of different lengths.
    """

    def __init__(
        self,
        vocab_size,
        emb_dim=128,
        num_filters=128,
        kernel_sizes=(3, 4, 5),
        out_dim=128,
        pad_idx=0,
        dropout=0.2,
    ):
        super().__init__()

        # Token embedding layer (padding_idx ensures PAD has zero gradient)
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        # Parallel 1D convolutions for different n-gram sizes
        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=emb_dim,
                out_channels=num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])

        # Regularization
        self.dropout = nn.Dropout(dropout)

        # Final projection to desired embedding dimension
        self.fc = nn.Linear(num_filters * len(kernel_sizes), out_dim)

    def forward(self, x):
        """
        x: Tensor [batch_size, seq_len]
        returns: Tensor [batch_size, out_dim] (L2-normalized)
        """

        # [B, T] → [B, T, E]
        emb = self.embedding(x)

        # Rearrange for Conv1d: [B, E, T]
        emb = emb.transpose(1, 2)

        # Apply convolution + ReLU + global max pooling
        conv_features = []
        for conv in self.convs:
            h = F.relu(conv(emb))                  # [B, F, T']
            h = F.max_pool1d(h, h.size(2))         # [B, F, 1]
            conv_features.append(h.squeeze(2))     # [B, F]

        # Concatenate features from all kernel sizes
        h = torch.cat(conv_features, dim=1)

        # Regularize + project
        h = self.dropout(h)
        h = self.fc(h)

        # Normalize to unit length (important for cosine similarity)
        return F.normalize(h, p=2, dim=1)

##### 5.5.2 Create Siamese Architecture

In [205]:
class SiameseCNN(nn.Module):
    """
    Siamese network for triplet-based metric learning.

    Applies a shared encoder to anchor (u), positive (v_pos),
    and negative (v_neg) inputs to obtain embeddings in the
    same latent space.
    """

    def __init__(self, encoder: nn.Module):
        super().__init__()

        # Shared feature extractor (weight sharing enforces
        # consistent representation learning)
        self.encoder = encoder

    def forward(self, u_tensor, v_pos_tensor, v_neg_tensor):
        """
        Inputs:
            u_tensor      : [B, T]
            v_pos_tensor  : [B, T]
            v_neg_tensor  : [B, T]

        Returns:
            emb_u, emb_pos, emb_neg : [B, D]
        """

        # Encode all three inputs using identical weights
        emb_u = self.encoder(u_tensor)
        emb_pos = self.encoder(v_pos_tensor)
        emb_neg = self.encoder(v_neg_tensor)

        return emb_u, emb_pos, emb_neg

##### 5.5.3 Model Instantiation & Device Allocation

In [206]:
# Automatically detect if a GPU is available for accelerated training
device = torch.device("mps" if torch.mps.is_available() else "cpu")

# Initialize the Feature Extractor (Encoder)
# We use a Multi-Kernel CNN to capture n-gram patterns of lengths 3, 4, and 5
encoder = TextCNNEncoder(
    vocab_size=len(vocab),  # Determined by the tokenizer in Section 6
    emb_dim=128,  # Dimensionality of the dense word vectors
    num_filters=128,  # Number of features to extract per kernel size
    kernel_sizes=(3, 4, 5),  # Window sizes: Tri-grams, 4-grams, 5-grams
    out_dim=128,  # Final embedding size (compact semantic vector)
    pad_idx=pad_id,  # Index to ignore during embedding lookup (zero gradient)
    dropout=0.4,  # Regularization to prevent overfitting on specific phrases
)

# Wrap the encoder in the Siamese architecture for metric learning
# scale=10.0 expands the cosine range [-1, 1] to [-10, 10] for sharper probability gradients
model = SiameseCNN(encoder).to(device)

print(f"Model initialized on: {device}")

Model initialized on: mps


### 5.6 Train the CNN

In [ ]:
# Triplet loss for metric learning.
# p=2 → Euclidean distance. Since embeddings are L2-normalized,
# minimizing Euclidean distance corresponds to optimizing angular separation.
criterion = nn.TripletMarginLoss(margin=1.0, p=2)

# AdamW optimizer with weight decay for regularization
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)


@torch.no_grad()
def eval_triplet_acc(model, loader):
    model.eval()
    correct = 0
    total = 0

    # Iterate over triplets (u, v_pos, v_neg)
    for _, _, _, u_tensor, v_pos_tensor, v_neg_tensor, _, _, _ in loader:
        u_tensor = u_tensor.to(device)
        v_pos_tensor = v_pos_tensor.to(device)
        v_neg_tensor = v_neg_tensor.to(device)

        # Forward pass through shared encoder
        emb_u, emb_v_pos, emb_v_neg = model(u_tensor, v_pos_tensor, v_neg_tensor)

        # Compute Euclidean distances in embedding space
        dist_pos = torch.norm(emb_u - emb_v_pos, p=2, dim=1)
        dist_neg = torch.norm(emb_u - emb_v_neg, p=2, dim=1)

        # Correct ranking if positive is closer than negative
        correct += (dist_pos < dist_neg).sum().item()
        total += u_tensor.size(0)

    return correct / total if total > 0 else 0.0


def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0

    for _, _, _, u_tensor, v_pos_tensor, v_neg_tensor, _, _, _ in loader:
        u_tensor = u_tensor.to(device)
        v_pos_tensor = v_pos_tensor.to(device)
        v_neg_tensor = v_neg_tensor.to(device)

        optimizer.zero_grad(set_to_none=True)

        # Forward pass through shared encoder
        emb_u, emb_v_pos, emb_v_neg = model(u_tensor, v_pos_tensor, v_neg_tensor)

        # Triplet loss:
        # Pull (u, v_pos) closer and push (u, v_neg) apart
        loss = criterion(emb_u, emb_v_pos, emb_v_neg)

        loss.backward()

        # Clip gradients to stabilize training
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item() * u_tensor.size(0)

    return total_loss / len(loader.dataset)


# Training loop
EPOCHS = 5
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model, train_loader)

    # Triplet accuracy = fraction of correctly ranked triplets
    test_acc = eval_triplet_acc(model, test_loader)
    train_acc = eval_triplet_acc(model, train_loader)

    print(
        f"Epoch {epoch:02d} | Loss: {loss:.4f} | "
        f"Train Acc: {train_acc:.2%} | Test Acc: {test_acc:.2%}"
    )

Epoch 01 | Loss: 0.9098 | Train Acc: 66.85% | Test Acc: 58.47%


### 5.7 Text Embedding Extraction

In [ ]:
@torch.no_grad()
def generate_user_embeddings_batched(model, unique_user_df, batch_size=256):
    """
    Generate L2-normalized user embeddings in batches using the trained encoder.
    
    Args:
        model: Trained Siamese model (encoder is used internally).
        unique_user_df: DataFrame with columns ["author", "body"].
        batch_size: Number of users processed per forward pass.
        
    Returns:
        user_dict: {user_id: embedding_vector}
        full_matrix: stacked embedding matrix aligned with user_ids
    """
    model.eval()

    # Preserve author order → required for alignment with embedding matrix
    user_ids = unique_user_df["author"].tolist()

    # Encode all user texts to token id sequences
    all_seqs = [encode(t) for t in unique_user_df["body"]]

    # Determine maximum sequence length for global padding
    max_len = max(len(s) for s in all_seqs)

    # Create padded tensor [N_users, max_len]
    padded_seqs = torch.full((len(all_seqs), max_len), pad_id, dtype=torch.long)
    for i, s in enumerate(all_seqs):
        padded_seqs[i, : len(s)] = torch.tensor(s)

    # Forward pass in mini-batches
    all_embs = []
    for i in range(0, len(padded_seqs), batch_size):
        batch = padded_seqs[i : i + batch_size].to(device)

        # Only encoder is needed (no triplet structure here)
        emb = model.encoder(batch)

        all_embs.append(emb.cpu().numpy())

    # Stack into final embedding matrix [N_users, D]
    full_matrix = np.vstack(all_embs)

    # Map user_id → embedding vector
    return {u_id: vec for u_id, vec in zip(user_ids, full_matrix)}, full_matrix


# --- Usage ---
# Aggregate latest comments per user (training split)
unique_users = (
    df_train.groupby("author")["body"]
    .apply(lambda s: " ".join(s.tail(10)))
    .reset_index()
)

user_vectors, embeddings_matrix = generate_user_embeddings_batched(model, unique_users)

# Alias for clarity
text_embeddings = user_vectors

# 6. Node2Vec for Graph-Based User Embeddings

In [ ]:
# ==========================================
# Construct Training Graph (Directed)
# ==========================================

# List of users with available text representations
user_id_list = unique_users["author"].tolist()

# Build directed interaction graph from training replies
# Edge u → v means: user u replied to user v
G = nx.DiGraph()

for u, v in pos_train[["u", "v"]].itertuples(index=False):
    G.add_edge(u, v)

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())


# ==========================================
# Learn Structural User Representations
# ==========================================

# Node2Vec performs biased random walks over the graph
# to capture structural proximity between users
node2vec = Node2Vec(
    G,
    dimensions=64,      # embedding dimensionality
    walk_length=30,     # length of each random walk
    num_walks=200,      # walks per node
    workers=1,          # single-threaded for reproducibility
    p=1.0,              # return parameter
    q=1.0,              # in-out parameter
    seed=42             # deterministic walks
)

# Train skip-gram model on generated walks
# Users appearing in similar walk contexts obtain similar vectors
n2v_model = Word2Vec(
    node2vec.walks,
    vector_size=64,
    window=10,
    min_count=1,
    batch_words=128,
    seed=42,
    workers=1,
)


# ==========================================
# Extract and Normalize Graph Embeddings
# ==========================================

graph_dim = 64
graph_embeddings = {}

for node in G.nodes():
    # Word2Vec stores keys as strings
    vec = n2v_model.wv[str(node)]

    # L2 normalization enables cosine similarity via dot product
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec = vec / norm

    graph_embeddings[node] = vec

Graph nodes: 999
Graph edges: 2722


Computing transition probabilities:   0%|          | 0/999 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|██████████| 200/200 [00:09<00:00, 20.91it/s]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


# 7. Joint Embeddings & FAISS Index Construction

In [ ]:
# ======================================================
# Build Joint Graph + Text Embedding Space
# ======================================================

joint_embeddings = {}
joint_vectors = []

# Keep only users for which BOTH structural and textual representations exist
common_users = [u for u in graph_embeddings if u in text_embeddings]

for u in common_users:
    g = graph_embeddings[u]   # structural embedding (Node2Vec)
    t = text_embeddings[u]    # textual embedding (CNN)

    # Normalize each modality independently
    # → prevents one modality from dominating due to scale differences
    g = g / (np.linalg.norm(g) + 1e-9)
    t = t / (np.linalg.norm(t) + 1e-9)

    # Concatenate structural + textual representations
    # Resulting vector captures both interaction patterns and language style
    z = np.concatenate([g, t])

    # Normalize the joint representation
    # → ensures cosine similarity via inner product
    z = z / (np.linalg.norm(z) + 1e-9)

    joint_embeddings[u] = z
    joint_vectors.append(z)


# ======================================================
# Build FAISS Index (Cosine Similarity Search)
# ======================================================

user_id_list = common_users
joint_matrix = np.vstack(joint_vectors).astype("float32")

# FAISS IndexFlatIP computes inner product
# With L2-normalized vectors, this equals cosine similarity
faiss.normalize_L2(joint_matrix)

d = joint_matrix.shape[1]
index = faiss.IndexFlatIP(d)
index.add(joint_matrix)

# Mapping from user ID → row in FAISS matrix
user_vectors = {u: joint_matrix[i] for i, u in enumerate(user_id_list)}

print("Joint Graph+Text FAISS index built with", index.ntotal, "users.")

Joint Graph+Text FAISS index built with 999 users.


# 8. Implement Retrieval Function

### 8.1 Implementation

In [ ]:
def build_candidate_pool(u):
    """
    Construct a structured candidate pool for user u.

    The pool is restricted to socially plausible users to reduce the
    search space before ranking.
    """
    candidates = set()

    # 1) Same-thread participants
    threads_u = set(pos_train[pos_train["u"] == u]["link_id"])
    for t in threads_u:
        users_in_thread = set(pos_train[pos_train["link_id"] == t]["u"])
        candidates.update(users_in_thread)

    # 2) Two-hop neighbors in the interaction graph
    if u in G:
        one_hop = set(G.neighbors(u))
        for n in one_hop:
            candidates.update(G.neighbors(n))

    # Remove invalid recommendations:
    # - already observed interactions in training
    # - the user themselves
    existing = set(pos_train[pos_train["u"] == u]["v"])
    candidates -= existing
    candidates.discard(u)

    return list(candidates)

### 8.2 Testing

In [ ]:
# Define test user
test_user = user_id_list[1]

# Build candidate pool for a specific test user
candidates = build_candidate_pool(test_user)

# Basic diagnostics
print("User:", test_user)
print("Number of candidates:", len(candidates))
print("First 10 candidates:", candidates[:10])

# Ensure the user is not recommending themselves
assert test_user not in candidates, "Self recommendation detected!"


# Collect already observed training interactions (u → v)
existing = set(pos_train[pos_train["u"] == test_user]["v"])

# Check whether candidate pool still contains known neighbors
overlap = set(candidates) & existing

print("Overlap with existing edges:", overlap)

# Ensure no previously seen edges are included
assert len(overlap) == 0, "Existing edges not removed!"

User: Thompson_S_Sweetback
Number of candidates: 1
First 10 candidates: ['Jaberkaty']
Overlap with existing edges: set()


# 9. Implement Score Function

In [ ]:
def score(u, v, alpha=0.5):
    graph_sim = np.dot(graph_embeddings[u], graph_embeddings[v])
    text_sim = np.dot(text_embeddings[u], text_embeddings[v])
    return alpha * graph_sim + (1 - alpha) * text_sim

# 10. Implement Hybrid Graph–Text Recommender

### 10.1 Implementation

In [ ]:
# Precompute training adjacency (u → set of interacted users)
train_edges = defaultdict(set)

for u, v in pos_train[["u", "v"]].itertuples(index=False):
    train_edges[u].add(v)  # store only outgoing interactions


def recommend_hybrid(u, k=10, fallback_search_k=200):
    """
    Two-stage recommendation:
    1) Structured candidate ranking (graph + text scoring)
    2) FAISS-based nearest neighbor fallback
    """

    # User must exist in embedding space
    if u not in graph_embeddings:
        return []

    # Users already interacted with in training (must be excluded)
    existing = train_edges.get(u, set())

    # ==================================================
    # Stage 1: Structured candidate retrieval + ranking
    # ==================================================
    candidates = build_candidate_pool(u)

    scored = []

    for v in candidates:

        # Ensure candidate has embeddings
        if v not in graph_embeddings:
            continue

        # Avoid recommending known neighbors
        if v in existing:
            continue

        # Hybrid similarity score (graph + text)
        s = score(u, v)
        scored.append((v, s))

    # Rank by descending similarity
    scored.sort(key=lambda x: x[1], reverse=True)
    recs = [v for v, _ in scored]

    # ======================================
    # Stage 2: FAISS similarity-based fallback
    # ======================================
    if len(recs) < k:

        # Query joint embedding (graph + text)
        query_vec = joint_embeddings[u].reshape(1, -1).astype("float32")

        # Retrieve top-N nearest neighbors (already sorted by similarity)
        distances, indices = index.search(query_vec, fallback_search_k)

        for idx in indices[0]:
            v = user_id_list[idx]

            # Exclusion rules
            if v == u:
                continue
            if v in existing:
                continue
            if v in recs:
                continue

            recs.append(v)

            if len(recs) == k:
                break

    # Return top-k recommendations
    return recs[:k]

### 10.2 Testing

In [ ]:
def test_recommendation(user_id):
    """
    Utility function to manually inspect top-k recommendations
    for a given user, including short text profile previews.
    """

    # Ensure the user exists in the embedding space
    if user_id not in user_vectors:
        return "User not found"

    # Generate top-3 recommendations using the CNN-based recommender
    recs = recommend_hybrid(user_id, k=3)

    # Print truncated history of the target user
    print(
        f"Target User ({user_id}) history sample: "
        f"{unique_users[unique_users['author'] == user_id]['body'].values[0][:100]}..."
    )
    print("-" * 30)

    # Print each recommended user with a short profile preview
    for i, rec_id in enumerate(recs):
        text = unique_users[
            unique_users["author"] == rec_id
        ]["body"].values[0][:100]

        print(f"Rec {i + 1}: {rec_id} | Text: {text}...")


# Example usage for quick qualitative inspection
test_recommendation(user_id_list[1])

Target User (Thompson_S_Sweetback) history sample: 1. Long term economic disincentives will not be very effective because people are naturally optimist...
------------------------------
Rec 1: Jaberkaty | Text: Excellent points. I would only add that most governments are going to try to add incentive to create...
Rec 2: EskimoNorth | Text: I didn't have to wear a uniform at school and I never once got bullied about what clothes I wore. Th...
Rec 3: RickyT44 | Text: I went to a uniform school where our uniform was a Polo with the school logo on it and a pair of kha...


In [ ]:
# Inspect the user ID at position 1 in the embedding index
print(user_id_list[1])

# Generate Top-10 recommendations for this user
recommend_hybrid(user_id_list[1])

Thompson_S_Sweetback


['Jaberkaty',
 'EskimoNorth',
 'RickyT44',
 'Rambleaway',
 'Bossman759',
 'elspazzz',
 'mideon2000',
 'Twonames',
 'Xenon_Iguana',
 'GreenSapphire']

# 11. Implement Echo Aware Popularity Re-Ranking Recommender

In [ ]:
def recommend_pop_echo_aware(user, k=10, lambda_penalty=0.5):
    """
    Popularity-based recommender with similarity penalty.
    Reduces echo chamber effects by penalizing highly similar users.
    """

    # User must exist in embedding space
    if user not in joint_embeddings:
        return []

    # Exclude already interacted users (training data)
    seen = set(pos_train[pos_train["u"] == user]["v"].values)

    # Normalize popularity to [0,1]
    max_pop = max(popularity_scores.values())

    scored = []

    # Iterate over globally popular users (strong exposure prior)
    for candidate in global_pop_ranking:

        # Skip invalid or already connected users
        if candidate == user or candidate in seen:
            continue
        if candidate not in joint_embeddings:
            continue

        # Popularity component (exposure strength)
        pop_score = popularity_scores.get(candidate, 0) / max_pop

        # Similarity component (echo signal)
        sim_score = np.dot(joint_embeddings[user], joint_embeddings[candidate])

        # Echo-aware scoring:
        # High similarity reduces the final score
        score = pop_score - lambda_penalty * sim_score

        scored.append((candidate, score))

        # Limit candidate pool for efficiency
        if len(scored) >= 300:
            break

    # Rank by final echo-aware score
    scored.sort(key=lambda x: x[1], reverse=True)

    return [v for v, _ in scored[:k]]

# 12. Evaluation

### 12.1 Build Ground Truth

In [ ]:
# 1. Build neighbor dictionaries
train_neighbors = pos_train.groupby("u")["v"].apply(set).to_dict()
test_neighbors = pos_test.groupby("u")["v"].apply(set).to_dict()

# 2. Users that exist in embedding index
embedded_users = set(user_vectors.keys())

ground_truth = {}

for u in test_neighbors:
    # Skip users without embeddings (cannot generate recommendations)
    if u not in embedded_users:
        continue

    train_set = train_neighbors.get(u, set())

    # Remove already seen interactions (only new links)
    new_interactions = test_neighbors[u] - train_set

    # Keep only targets that also have embeddings
    new_interactions = {v for v in new_interactions if v in embedded_users}

    # Only keep users with at least one evaluable target
    if len(new_interactions) > 0:
        ground_truth[u] = new_interactions

### 12.2 Evaluate Precision@K

In [ ]:
def evaluate_precision_at_k(model_recommend_fn, ground_truth, k=10):
    """
    Computes macro-averaged Precision@K.

    Args:
        model_recommend_fn: function(u, k) -> ranked list of recommended users
        ground_truth: dict {u: set of relevant future interaction targets}
        k: cutoff rank
    """
    total_hits = 0
    total_users = 0

    for u, true_targets in ground_truth.items():

        # Top-K recommendations for user u
        recs = model_recommend_fn(u, k=k)

        # Keep only users that exist in the embedding/index space
        recs = [r for r in recs if r in user_vectors]

        # Number of correctly recommended future interactions
        hits = len(set(recs) & true_targets)

        total_hits += hits
        total_users += 1

    if total_users == 0:
        return 0.0

    # Macro-averaged Precision@K:
    # (Total correct recommendations) / (K × number of evaluated users)
    return total_hits / (k * total_users)

In [ ]:
precision_random = evaluate_precision_at_k(
    recommend_random, ground_truth, k=10)
precision_popularity = evaluate_precision_at_k(
    recommend_popularity, ground_truth, k=10)
precision_common_neighbors = evaluate_precision_at_k(
    recommend_common_neighbors, ground_truth, k=10)
precision_hybrid = evaluate_precision_at_k(
    recommend_hybrid, ground_truth, k=10)
precision_pop_ea = evaluate_precision_at_k(
    recommend_pop_echo_aware, ground_truth, k=10)

print("Random Baseline Precision@10:", precision_random)
print("Popularity Baseline Precision@10:", precision_popularity)
print("Common Neighbors Baseline Precision@10:", precision_common_neighbors)
print("Hybrid Precision@10:", precision_hybrid)
print("Poppularity Echo Aware Precision@10:", precision_pop_ea)

Random Baseline Precision@10: 0.0030303030303030303
Popularity Baseline Precision@10: 0.03484848484848485
Common Neighbors Baseline Precision@10: 0.019696969696969695
Hybrid Precision@10: 0.0015151515151515152
Poppularity Echo Aware Precision@10: 0.024242424242424242


### 12.3 Evaluate Recall@K

In [ ]:
def evaluate_recall_at_k(model_recommend_fn, ground_truth, k=10):
    """
    Computes macro-averaged Recall@K.

    Args:
        model_recommend_fn: function(u, k) -> ranked list of recommended users
        ground_truth: dict {u: set of relevant future interaction targets}
        k: cutoff rank
    """

    total_recall = 0.0
    total_users = 0

    for u, true_targets in ground_truth.items():

        # Top-K recommendations for user u
        recs = model_recommend_fn(u, k=k)

        # Keep only users that exist in the embedding/index space
        recs = [r for r in recs if r in user_vectors]

        # Number of correctly recommended future interactions
        hits = len(set(recs) & true_targets)

        # User-level recall:
        # fraction of relevant targets recovered in Top-K
        recall_u = hits / len(true_targets)

        total_recall += recall_u
        total_users += 1

    if total_users == 0:
        return 0.0

    # Macro-averaged Recall@K:
    # average of per-user recall values
    return total_recall / total_users

In [ ]:
recall_random = evaluate_recall_at_k(recommend_random, ground_truth, k=10)

recall_popularity = evaluate_recall_at_k(
    recommend_popularity, ground_truth, k=10)

recall_common_neighbors = evaluate_recall_at_k(
    recommend_common_neighbors, ground_truth, k=10)

recall_hybrid = evaluate_recall_at_k(recommend_hybrid, ground_truth, k=10)

recall_pop_ea = evaluate_recall_at_k(
    recommend_pop_echo_aware, ground_truth, k=10)

print("Random Baseline Recall@10:", recall_random)
print("Popularity Baseline Recall@10:", recall_popularity)
print("Common Neighbors Baseline Recall@10:", recall_common_neighbors)
print("Hybrid Recall@10:", recall_hybrid)
print("Popularity Echo Aware Recall@10:", recall_pop_ea)

Random Baseline Recall@10: 0.0
Popularity Baseline Recall@10: 0.21153854562945476
Common Neighbors Baseline Recall@10: 0.0928221610039792
Hybrid Recall@10: 0.007575757575757576
Popularity Echo Aware Recall@10: 0.16843434343434344


### 12.4 Evaluate nDCG@K

In [ ]:
def evaluate_ndcg_at_k(model_recommend_fn, ground_truth, k=10):
    """
    Computes macro-averaged nDCG@K (Normalized Discounted Cumulative Gain).

    Args:
        model_recommend_fn: function(u, k) -> ranked list of recommended users
        ground_truth: dict {u: set of relevant future interaction targets}
        k: cutoff rank
    """

    total_ndcg = 0.0
    total_users = 0

    for u, true_targets in ground_truth.items():

        # Top-K ranked recommendations for user u
        recs = model_recommend_fn(u, k=k)

        # -----------------------
        # 1) DCG (Discounted Gain)
        # -----------------------
        # Rewards relevant items appearing at higher ranks.
        # Log-discount penalizes lower-ranked hits.
        dcg = 0.0
        for rank, candidate in enumerate(recs, start=1):
            if candidate in true_targets:
                dcg += 1.0 / np.log2(rank + 1)

        # -----------------------
        # 2) IDCG (Ideal DCG)
        # -----------------------
        # Maximum possible DCG if all relevant items were perfectly ranked.
        ideal_hits = min(len(true_targets), k)
        idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))

        # Skip users without evaluable ground truth
        if idcg == 0:
            continue

        # -----------------------
        # 3) Normalization
        # -----------------------
        # nDCG_u ∈ [0, 1]
        ndcg_u = dcg / idcg

        total_ndcg += ndcg_u
        total_users += 1

    if total_users == 0:
        return 0.0

    # Macro-average over users
    return total_ndcg / total_users

In [ ]:
ndcg_random = evaluate_ndcg_at_k(recommend_random, ground_truth, k=10)

ndcg_popularity = evaluate_ndcg_at_k(recommend_popularity, ground_truth, k=10)

ndcg_common_neighbors = evaluate_ndcg_at_k(
    recommend_common_neighbors, ground_truth, k=10)

ndcg_hybrid = evaluate_ndcg_at_k(recommend_hybrid, ground_truth, k=10)

ndcg_pop_ea = evaluate_ndcg_at_k(recommend_pop_echo_aware, ground_truth, k=10)

print("Random Baseline nDCG@10:", ndcg_random)
print("Popularity Baseline nDCG@10:", ndcg_popularity)
print("Common Neighbors Baseline nDCG@10:", ndcg_common_neighbors)
print("Hybrid nDCG@10:", ndcg_hybrid)
print("Popularity Echo Aware nDCG@10:", ndcg_pop_ea)

Random Baseline nDCG@10: 0.0
Popularity Baseline nDCG@10: 0.18155685015369388
Common Neighbors Baseline nDCG@10: 0.06869534591141693
Hybrid nDCG@10: 0.005861406170220328
Popularity Echo Aware nDCG@10: 0.16384730862071495


### 12.5 Evaluate Echo Chamber Metrics

##### 12.5.1 Unified Distance Metric Construction (Graph + Text Space)

In [ ]:
# ======================================================
# Build Unified User Representation for Distance Metric
# ======================================================

dm_map = {}

for user in user_id_list:

    # Only keep users that have both modalities available
    if user in graph_embeddings and user in text_embeddings:

        # --- Normalize each modality separately ---
        # Ensures graph and text contribute equally
        g_norm = graph_embeddings[user] / (
            np.linalg.norm(graph_embeddings[user]) + 1e-9
        )

        t_norm = text_embeddings[user] / (
            np.linalg.norm(text_embeddings[user]) + 1e-9
        )

        # --- Concatenate graph + text vectors ---
        joint_vec = np.concatenate([g_norm, t_norm])

        # --- Final normalization ---
        # Places all users on the same hypersphere
        dm_map[user] = joint_vec / np.linalg.norm(joint_vec)


# ======================================================
# Distance Metric (Euclidean in Joint Space)
# ======================================================

def calculate_dm_distance(u, v):
    """
    Euclidean distance in the joint graph-text embedding space.
    Used for diversity and novelty evaluation.
    """
    return np.linalg.norm(dm_map[u] - dm_map[v])

# ======================================================
# Precompute Training Edges (Fast Lookup)
# ======================================================

train_edges = defaultdict(set)

for u, v in pos_train[["u", "v"]].itertuples(index=False):
    # Store outgoing interactions only
    train_edges[u].add(v)

##### 12.5.2 Evaluate Individual Diversity@K

In [ ]:
def individual_diversity(recs):
    """
    Computes the average pairwise distance between all recommended users.

    A higher value indicates that the recommendation list
    contains users that are more dissimilar to each other.
    """

    # Compute joint-space distances for all ordered user pairs (i ≠ j)
    dists = [
        calculate_dm_distance(i, j)
        for i in recs
        for j in recs
        if i != j
    ]

    # Return mean distance (0 if list too small)
    return np.mean(dists) if dists else 0.0

In [ ]:
def evaluate_individual_diversity_at_k(model_recommend_fn, k=10):
    """
    Computes the average individual diversity@k across all evaluable users.

    For each user:
    - Generate top-k recommendations
    - Compute the mean pairwise distance (DM) among the recommended users
    - Average this value across users
    """

    total_div = 0.0
    total_users = 0

    for u in ground_truth.keys():

        # Generate ranked Top-k list
        recs = model_recommend_fn(u, k=k)

        # Diversity requires at least two items
        if len(recs) < 2:
            continue

        dists = []

        # Compute pairwise distances (upper triangular only, no duplicates)
        for i in range(len(recs)):
            for j in range(i + 1, len(recs)):

                # Ensure both users exist in joint embedding space
                if recs[i] in dm_map and recs[j] in dm_map:
                    d = calculate_dm_distance(recs[i], recs[j])
                    dists.append(d)

        # Skip if no valid distance could be computed
        if len(dists) == 0:
            continue

        # Average diversity for this user
        total_div += np.mean(dists)
        total_users += 1

    # Return mean diversity across users
    return total_div / total_users if total_users > 0 else 0.0

In [ ]:
indiv_div_random = evaluate_individual_diversity_at_k(recommend_random, k=10)

indiv_div_popularity = evaluate_individual_diversity_at_k(
    recommend_popularity, k=10)

indiv_div_common_neighbors = evaluate_individual_diversity_at_k(
    recommend_common_neighbors, k=10)

indiv_div_hybrid = evaluate_individual_diversity_at_k(recommend_hybrid, k=10)

indiv_div_pop_ea = evaluate_individual_diversity_at_k(
    recommend_pop_echo_aware, k=10)

print("Random Baseline Individual Diversity@10:", indiv_div_random)
print("Popularity Baseline Individual Diversity@10:", indiv_div_popularity)
print("Common Neighbors Baseline Individual Diversity@10:",
      indiv_div_common_neighbors)
print("Hybrid Individual Diversity@10:", indiv_div_hybrid)
print("Popularity Echo Aware Individual Diversity@10:", indiv_div_pop_ea)

Random Baseline Individual Diversity@10: 1.2310314458428007
Popularity Baseline Individual Diversity@10: 1.000050529386058
Common Neighbors Baseline Individual Diversity@10: 1.0932372093200684
Hybrid Individual Diversity@10: 0.9089002175764604
Popularity Echo Aware Individual Diversity@10: 1.2451628627199116


##### 12.5.3 Evaluate Individual Novelty@K

In [ ]:
def individual_novelty(recs, existing_interactions):
    """
    Computes the novelty of a recommendation list for a single user.

    Novelty is defined as the average embedding distance (DM)
    between each recommended user and the user's existing
    training interactions (Fu).

    Higher values indicate that recommended users are more
    dissimilar from previously interacted users.
    """

    # Compute distances between each recommended user (r)
    # and each previously interacted user (f ∈ Fu)
    dists = [
        calculate_dm_distance(r, f)
        for r in recs
        for f in existing_interactions
        if r in dm_map and f in dm_map
    ]

    # Return mean distance; 0 if no valid comparisons exist
    return np.mean(dists) if dists else 0.0

In [ ]:
def evaluate_individual_novelty_at_k(model_recommend_fn, k=10):
    """
    Computes average individual novelty@k across all evaluable users.

    For each user:
        - Generate top-k recommendations
        - Compare them to the user's historical training neighbors (F_u)
        - Measure average embedding distance (DM)

    Higher values indicate more novel (less similar) recommendations.
    """

    total_novelty = 0.0
    total_users = 0

    for u in ground_truth.keys():
        # Generate ranked recommendations
        recs = model_recommend_fn(u, k=k)

        # Historical neighbors from training (F_u)
        F_u = train_edges.get(u, set())

        # Skip users without recommendations or without interaction history
        if len(recs) == 0 or len(F_u) == 0:
            continue

        dists = []

        # Compute distances between each recommended user r
        # and each previously interacted user f ∈ F_u
        for r in recs:
            if r not in dm_map:
                continue
            for f in F_u:
                if f not in dm_map:
                    continue
                d = calculate_dm_distance(r, f)
                dists.append(d)

        # Skip if no valid embedding comparisons were possible
        if len(dists) == 0:
            continue

        # Average novelty for this user
        total_novelty += np.mean(dists)
        total_users += 1

    # Return macro-average novelty across users
    return total_novelty / total_users if total_users > 0 else 0.0

In [ ]:
indiv_nov_random = evaluate_individual_novelty_at_k(recommend_random, k=10)

indiv_nov_popularity = evaluate_individual_novelty_at_k(
    recommend_popularity, k=10)

indiv_nov_common_neighbors = evaluate_individual_novelty_at_k(
    recommend_common_neighbors, k=10)

indiv_nov_hybrid = evaluate_individual_novelty_at_k(recommend_hybrid, k=10)

indiv_nov_pop_ea = evaluate_individual_novelty_at_k(
    recommend_pop_echo_aware, k=10)

print("Random Baseline Individual Novelty@10:", indiv_nov_random)
print("Popularity Baseline Individual Novelty@10:", indiv_nov_popularity)
print("Common Neighbors Baseline Individual Novelty@10:",
      indiv_nov_common_neighbors)
print("Hybrid Individual Novelty@10:", indiv_nov_hybrid)
print("Popularity Echo Aware Individual Novelty@10:", indiv_nov_pop_ea)

Random Baseline Individual Novelty@10: 1.2167616952210665
Popularity Baseline Individual Novelty@10: 1.1066786805167794
Common Neighbors Baseline Individual Novelty@10: 1.1020793337670585
Hybrid Individual Novelty@10: 1.0596961649134755
Popularity Echo Aware Individual Novelty@10: 1.3203203175216913


##### 12.5.4 Community Detection via Louvain

In [ ]:
# --------------------------------------------
# Build Undirected Weighted Interaction Graph
# --------------------------------------------

# Create undirected graph:
# Direction is ignored here because we want structural communities,
# not reply directionality.
G = nx.Graph()

# Aggregate interaction frequency between user pairs
# Each (u, v) edge receives a weight equal to the number of replies.
edge_weights = (
    pos_train
    .groupby(["u", "v"])
    .size()
    .reset_index(name="weight")
)

# Add weighted edges to the graph
for u, v, w in edge_weights.itertuples(index=False):
    if G.has_edge(u, v):
        # If edge already exists, accumulate weight
        G[u][v]["weight"] += w
    else:
        # Otherwise create new weighted edge
        G.add_edge(u, v, weight=w)


# --------------------------------------------
# Community Detection (Louvain Method)
# --------------------------------------------

# Apply Louvain algorithm to detect densely connected groups.
# Output: dictionary mapping each user to a community ID.
partition = community_louvain.best_partition(G)

# Organize users by community for fast lookup
# community_members: {community_id -> set(users)}
community_members = defaultdict(set)

for user, comm in partition.items():
    community_members[comm].add(user)

##### 12.5.5 Evaluate Community Diversity@K

In [ ]:
def community_diversity(recommend_fn, k=10):
    """
    Community Diversity:
    Measures how diverse the union of recommendations is within each community.
    
    For every detected community:
        1. Collect the Top-k recommendations for all its members.
        2. Merge them into one set R_c.
        3. Compute the average pairwise distance in DM space inside R_c.
    
    The final score is the average diversity across communities.
    """

    community_scores = []

    # Iterate over each detected community
    for comm, members in community_members.items():

        # R_c = union of recommendations for all users in community c
        R_c = set()

        for u in members:
            if u in user_vectors:
                R_c.update(recommend_fn(u, k=k))

        R_c = list(R_c)

        # Need at least two users to compute pairwise distances
        if len(R_c) < 2:
            continue

        # Compute all pairwise DM distances inside R_c
        dists = [
            calculate_dm_distance(i, j)
            for i in R_c
            for j in R_c
            if i != j
        ]

        # Store average community-level diversity
        if dists:
            community_scores.append(np.mean(dists))

    # Average over all communities
    return np.mean(community_scores) if community_scores else 0.0

In [ ]:
comm_div_random = community_diversity(recommend_random, k=10)

comm_div_popularity = community_diversity(recommend_popularity, k=10)

comm_div_common_neighbors = community_diversity(recommend_common_neighbors, k=10)

comm_div_hybrid = community_diversity(recommend_hybrid, k=10)

comm_div_pop_ea = community_diversity(recommend_pop_echo_aware, k=10)

print("Random Baseline Community Diversity@10:", comm_div_random)
print("Popularity Baseline Community Diversity@10:", comm_div_popularity)
print("Common Neighbors Community Diversity@10:", comm_div_common_neighbors)
print("Hybrid Community Diversity@10:", comm_div_hybrid)
print("Popularity Echo Aware Community Diversity@10:", comm_div_pop_ea)

Random Baseline Community Diversity@10: 1.2332081
Popularity Baseline Community Diversity@10: 0.9976331
Common Neighbors Community Diversity@10: 1.1437206
Hybrid Community Diversity@10: 1.0871369
Popularity Echo Aware Community Diversity@10: 1.3066009


##### 12.5.6 Evaluate Community Novelty@K

In [ ]:
def community_novelty(recommend_fn, k=10):
    """
    Community Novelty:
    Measures how different the recommended users are from the
    historical interaction neighborhood of the entire community.

    For every detected community:
        1. R_c = union of Top-k recommendations of all members.
        2. F_c = union of all historical training neighbors of members.
        3. Compute average DM distance between R_c and F_c.
    
    The final score is averaged across all communities.
    """

    community_scores = []

    # Iterate over each detected community
    for comm, members in community_members.items():

        R_c = set()  # union of recommendations
        F_c = set()  # union of historical neighbors

        for u in members:
            if u in user_vectors:
                # Collect recommendations
                R_c.update(recommend_fn(u, k=k))

                # Collect historical interactions (training graph)
                F_c.update(train_edges.get(u, set()))

        R_c = list(R_c)
        F_c = list(F_c)

        # Need both recommended and historical users to compute novelty
        if len(R_c) == 0 or len(F_c) == 0:
            continue

        # Compute cross-distance between recommendations and past interactions
        dists = [
            calculate_dm_distance(i, j)
            for i in R_c
            for j in F_c
        ]

        # Store average novelty for this community
        if dists:
            community_scores.append(np.mean(dists))

    # Average across communities
    return np.mean(community_scores) if community_scores else 0.0

In [ ]:
comm_nov_random = community_novelty(recommend_random, k=10)

comm_nov_popularity = community_novelty(recommend_popularity, k=10)

comm_nov_common_neighbors = community_novelty(recommend_common_neighbors, k=10)

comm_nov_hybrid = community_novelty(recommend_hybrid, k=10)

comm_nov_pop_ea = community_novelty(recommend_pop_echo_aware, k=10)

print("Random Baseline Community Novelty@10:", comm_nov_random)
print("Popularity Baseline Community Novelty@10:", comm_nov_popularity)
print("Common Neighbors Community Novelty@10:", comm_nov_common_neighbors)
print("Hybrid Community Novelty@10:", comm_nov_hybrid)
print("Popularity Echo Aware Community Novelty@10:", comm_nov_pop_ea)

Random Baseline Community Novelty@10: 1.2091303
Popularity Baseline Community Novelty@10: 1.158631
Common Neighbors Community Novelty@10: 1.0879058
Hybrid Community Novelty@10: 1.0378795
Popularity Echo Aware Community Novelty@10: 1.3687328


##### 12.5.7 Evaluate Cross Community Exposure Rate@K

In [ ]:
def cross_community_rate_at_k(u, recommend_fn, k=10):
    """
    Cross-Community Exposure Rate@K (per user).

    Measures the proportion of Top-K recommendations that belong
    to a different community than the target user.

    Args:
        u: target user
        recommend_fn: function(u, k) -> ranked recommendation list
        k: evaluation cutoff

    Returns:
        Fraction of cross-community recommendations in [0, 1],
        or None if the user cannot be evaluated.
    """

    # User must have an assigned community (from Louvain partition)
    if u not in partition:
        return None

    user_comm = partition[u]

    # Generate Top-K recommendations
    recs = recommend_fn(u, k=k)

    # Cannot evaluate if no recommendations were produced
    if len(recs) == 0:
        return None

    cross_count = 0

    # Count how many recommended users belong to a different community
    for v in recs:
        if v in partition and partition[v] != user_comm:
            cross_count += 1

    # Normalize by K (fixed exposure budget)
    return cross_count / k

In [ ]:
def evaluate_cross_community_exposure_at_k(recommend_fn, k=10):
    """
    Macro-averaged Cross-Community Exposure@K.

    Computes the average fraction of cross-community recommendations
    across all evaluable users.

    Args:
        recommend_fn: function(u, k) -> ranked recommendations
        k: evaluation cutoff

    Returns:
        Mean cross-community exposure in [0, 1].
    """

    total_score = 0.0
    total_users = 0

    # Iterate over evaluable users (same population as ranking metrics)
    for u in ground_truth.keys():

        # User must have a detected community
        if u not in partition:
            continue

        # Per-user cross-community exposure
        score = cross_community_rate_at_k(u, recommend_fn, k=k)

        # Skip users that cannot be evaluated
        if score is None:
            continue

        total_score += score
        total_users += 1

    # Macro average across users
    return total_score / total_users if total_users > 0 else 0.0

In [ ]:
cross_random = evaluate_cross_community_exposure_at_k(recommend_random, k=10)
cross_popularity = evaluate_cross_community_exposure_at_k(recommend_popularity, k=10)
cross_common_neighbors = evaluate_cross_community_exposure_at_k(recommend_common_neighbors, k=10)
cross_hybrid = evaluate_cross_community_exposure_at_k(recommend_hybrid, k=10)
cross_pop_ea = evaluate_cross_community_exposure_at_k(recommend_pop_echo_aware, k=10)

print("Random Cross-Community@10:", cross_random)
print("Popularity Cross-Community@10:", cross_popularity)
print("Common Neighbors Cross-Community@10:", cross_common_neighbors)
print("Hybrid Cross-Community@10:", cross_hybrid)
print("Popularity Echo-Aware Cross-Community@10:", cross_pop_ea)

Random Cross-Community@10: 0.9469696969696966
Popularity Cross-Community@10: 0.9636363636363632
Common Neighbors Cross-Community@10: 0.6738461538461539
Hybrid Cross-Community@10: 0.6136363636363635
Popularity Echo-Aware Cross-Community@10: 0.9772727272727271
